# DVC (Data Version Control)

A refresher on **DVC** — *Git for data, models, and ML pipelines*. DVC keeps your
big files **out** of Git: it stores a tiny text pointer in Git and the real bytes in
a content-addressed cache (and on remote storage), so you can version a 5 GB dataset
the same way you version a 50-line script. On top of that it adds **reproducible
pipelines** (`dvc.yaml`) and lightweight **experiment/metric tracking**.

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes  ·  _runs fully offline; no cloud account needed_

## 1. What & Why

**What it is.** A command-line tool (with a Python API) that layers data versioning
on top of Git. Git is great at text and terrible at large binaries — it stores every
version inline, bloating the repo forever. DVC fixes this: you run `dvc add data/`
and DVC moves the data into a local **cache**, computes its hash, and writes a small
`*.dvc` pointer file (just the hash + path). **Git tracks the pointer; DVC tracks the
data.** `git checkout` switches code *and* pointers; `dvc checkout` then materializes
the matching data from the cache. The actual bytes live on a **remote** (S3, GCS,
Azure, SSH, or just another folder) that you `dvc push`/`dvc pull` like `git
push`/`pull`.

Beyond versioning, DVC adds two things teams need:

- **Pipelines** — a `dvc.yaml` of *stages* (command + dependencies + outputs). `dvc
  repro` runs only the stages whose inputs changed and caches the rest, so your
  `prepare → train → evaluate` DAG is reproducible and incremental.
- **Metrics & experiments** — track metric files across runs (`dvc metrics show`,
  `dvc exp run`) to compare model variants without a server.

**The problem it solves.** "Which data and which code produced this model?" Without
DVC, datasets live in someone's Dropbox, model weights are emailed around, and the
commit that trained `model_final_v3_REAL.pkl` is lost. DVC makes data, code, and the
pipeline that connects them a single, reproducible, `git checkout`-able unit.

**Reach for it when** datasets/models are too big for Git, you need reproducible
data pipelines, or a team must share large artifacts with versioning and lineage.
**Skip it when** all your data is small enough to commit directly, or you've already
standardized on a heavier platform (a feature store, LakeFS, or a managed MLOps
suite) that covers the same ground.

## 2. Mental Model

**Git stores a *receipt*; DVC stores the *package*.** When you `dvc add bigfile`,
the bytes leave your working tree's Git history and go to a content-addressed cache;
Git only ever sees a tiny `bigfile.dvc` text file containing the hash. Checking out a
commit restores the *pointers*; `dvc checkout` (or `dvc pull`) then hydrates the real
files to match.

```
   WORKING DIR            GIT (tiny)              DVC CACHE / REMOTE (big)
 ┌──────────────┐       ┌──────────────┐        ┌───────────────────────────┐
 │ data.csv  ●──┼──────▶│ data.csv.dvc │        │ files/md5/5f/a41a28...     │ ◀── content-
 │ model.pkl ●──┼──────▶│ model.pkl.dvc│  hash  │ files/md5/9c/3b07f1...     │     addressed
 │ src/train.py │──────▶│ src/train.py │ ─────▶ │            ▲               │     by hash
 │ dvc.yaml     │──────▶│ dvc.yaml     │        │  dvc push  │  dvc pull     │
 │ dvc.lock     │──────▶│ dvc.lock     │        └────────────┼──────────────┘
 └──────────────┘       └──────────────┘                S3 / GCS / SSH / dir
        ▲ the ● files are git-ignored; only the .dvc pointers are committed
```

The mental shortcut: **a `.dvc` file (or a `dvc.yaml` output) is a symlink-by-hash.**
Commit code + pointers to Git, push the bytes to a DVC remote, and anyone can
`git pull && dvc pull` to reconstruct the exact data/model for that commit. A
**pipeline** is the same idea applied to *commands*: `dvc.lock` records the hash of
every input and output, so `dvc repro` re-runs a stage only when its hashes change.

## 3. Key Concepts

- **`.dvc` file** — a small YAML pointer (`md5`, `size`, `path`) committed to Git in
  place of the real data. One per tracked file/dir added with `dvc add`.
- **Cache** (`.dvc/cache`) — local content-addressed store keyed by hash. Tracked
  files are *links* (reflink/hardlink/symlink) into it, so duplicates cost no extra
  space and switching versions is instant.
- **Remote** — off-machine storage (`s3://`, `gs://`, `ssh://`, or a local dir) for
  the cache. `dvc remote add -d`, then `dvc push` / `dvc pull` mirror `git push/pull`.
- **Stage** — one step of a pipeline: a `cmd` plus its **deps** (`-d`) and **outs**
  (`-o`). Declared in `dvc.yaml`.
- **`dvc.yaml` / `dvc.lock`** — the pipeline definition (human-written) and its lock
  file (machine-written hashes of every dep/out). Both are committed to Git.
- **`dvc repro`** — runs the pipeline DAG, executing only stages whose dependency
  hashes changed; everything else is restored from cache. Incremental + reproducible.
- **`dvc checkout` / `dvc pull`** — restore working-tree data to match the committed
  pointers, from the cache or the remote respectively.
- **Metrics & plots** — small files (`-M metrics.json`) DVC reads to tabulate/compare
  runs (`dvc metrics show`, `dvc metrics diff`).
- **Experiments** (`dvc exp run` / `exp show`) — lightweight, Git-backed parameter
  sweeps that don't clutter your branch history until you promote one.
- **`dvc.api`** — Python API (`dvc.api.read`, `open`, `get_url`) to read DVC-tracked
  data straight from a repo without manually checking it out.

## 4. Setup

DVC is a Python package; install it with the right extra for your remote
(`dvc[s3]`, `dvc[gs]`, `dvc[ssh]`, …). It needs **Git** in the project — DVC stores
its pointers in your Git repo and refuses to init outside one (use `dvc init
--no-scm` only for throwaway cases).

This notebook drives DVC entirely inside a **temporary Git repo in a temp directory**
so it touches neither your real repo nor the network. The "remote" is just another
local folder — the exact same `dvc push` workflow you'd use with S3, minus the
credentials.

In [1]:
# On a fresh environment, uncomment to install (pick the extra for your remote):
# %pip install -q "dvc"            # core; add dvc[s3] / dvc[gs] / dvc[ssh] for clouds

import os, sys, subprocess, tempfile, textwrap
from pathlib import Path

# A scratch Git repo so nothing here touches your real project or the network.
WORK = Path(tempfile.mkdtemp(prefix="dvc-refresher-"))
REMOTE = Path(tempfile.mkdtemp(prefix="dvc-remote-"))   # stands in for s3:// etc.

def sh(*args, cwd=WORK):
    """Run a command in the scratch repo and return its stdout (errors raise)."""
    out = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if out.returncode != 0:
        raise RuntimeError(out.stderr.strip() or out.stdout.strip())
    return out.stdout.strip()

def dvc(*args):
    """Invoke the DVC CLI via the current interpreter (PATH-independent)."""
    return sh(sys.executable, "-m", "dvc", "-q", *args)

# Git is a prerequisite; configure a local identity so commits/init are clean.
sh("git", "init", "-q")
sh("git", "config", "user.email", "refresher@example.com")
sh("git", "config", "user.name", "DVC Refresher")
dvc("init")                                   # creates .dvc/ and .dvcignore

print("dvc     ", dvc("--version"))
print("workdir ", WORK)
print("scaffold", sorted(p.name for p in WORK.iterdir()))

dvc      3.67.1
workdir  /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/dvc-refresher-__9bnif8
scaffold ['.dvc', '.dvcignore', '.git']


## 5. Worked Examples

### Example 1 — Version a data file with `dvc add`

The core move. We create a "dataset", `dvc add` it, and watch DVC (a) write a tiny
`*.dvc` pointer for Git to track, (b) auto-append the real file to `.gitignore`, and
(c) copy the bytes into the content-addressed cache. The pointer's `md5` *is* the
cache key.

In [2]:
# Create a small "dataset" and hand it to DVC.
data = WORK / "data.csv"
data.write_text("id,x\n1,10\n2,20\n3,30\n")

dvc("add", "data.csv")          # -> data.csv.dvc + cache entry + .gitignore line

print("--- data.csv.dvc (this is what Git commits) ---")
print((WORK / "data.csv.dvc").read_text())
print("--- .gitignore (DVC excluded the real file from Git) ---")
print((WORK / ".gitignore").read_text().strip())

cache_files = [p for p in (WORK / ".dvc/cache").rglob("*") if p.is_file()]
print("--- cache (content-addressed by md5) ---")
for p in cache_files:
    print(p.relative_to(WORK))

# Commit *the pointer*, never the data, to Git.
sh("git", "add", "data.csv.dvc", ".gitignore")
sh("git", "commit", "-q", "-m", "Track data.csv with DVC")
print("\ngit now tracks:", ", ".join(sh("git", "ls-files").splitlines()))

--- data.csv.dvc (this is what Git commits) ---
outs:
- md5: 5fa41a28661ee5ba0bbd1c6d21267325
  size: 20
  hash: md5
  path: data.csv

--- .gitignore (DVC excluded the real file from Git) ---
/data.csv
--- cache (content-addressed by md5) ---
.dvc/cache/files/md5/5f/a41a28661ee5ba0bbd1c6d21267325

git now tracks: .dvc/.gitignore, .dvc/config, .dvcignore, .gitignore, data.csv.dvc


### Example 2 — A reproducible pipeline (`dvc.yaml` + `dvc repro`)

A *stage* binds a command to its dependencies and outputs. `dvc repro` runs the DAG
but **skips stages whose inputs haven't changed** — the heart of reproducible,
incremental ML. We add a stage that sums the data, run it, then run it again to see
DVC cache the result. The `-M` output is a *metrics* file DVC can tabulate.

In [3]:
# A trivial "training" script: read the data, write an output + a metrics file.
(WORK / "process.py").write_text(textwrap.dedent("""\
    import csv, json
    rows = list(csv.DictReader(open("data.csv")))
    total = sum(int(r["x"]) for r in rows)
    open("sum.txt", "w").write(str(total))
    json.dump({"rows": len(rows), "total": total}, open("metrics.json", "w"))
"""))

# Define the stage: cmd + deps (-d) + output (-o) + metrics (-M).
dvc("stage", "add", "-n", "process",
    "-d", "process.py", "-d", "data.csv",
    "-o", "sum.txt", "-M", "metrics.json",
    f"{sys.executable} process.py")

print("=== first repro (runs the stage) ===")
print(dvc("repro"))
print("\nsum.txt =>", (WORK / "sum.txt").read_text())
print("metrics =>", dvc("metrics", "show"))

print("\n=== second repro (nothing changed -> cached/skipped) ===")
print(dvc("repro") or "Data and pipelines are up to date.")

# dvc.lock pins the exact hash of every dependency and output.
print("\n--- dvc.lock (committed alongside dvc.yaml) ---")
print((WORK / "dvc.lock").read_text())

=== first repro (runs the stage) ===


'data.csv.dvc' didn't change, skipping
Running stage 'process':
> /Users/danieldekerlegand/Development/ai-tutor/.venv/bin/python3.13 process.py
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.

sum.txt => 60


metrics => Path          rows    total
metrics.json  3       60

=== second repro (nothing changed -> cached/skipped) ===


'data.csv.dvc' didn't change, skipping
Stage 'process' didn't change, skipping
Data and pipelines are up to date.

--- dvc.lock (committed alongside dvc.yaml) ---
schema: '2.0'
stages:
  process:
    cmd: /Users/danieldekerlegand/Development/ai-tutor/.venv/bin/python3.13 
      process.py
    deps:
    - path: data.csv
      hash: md5
      md5: 5fa41a28661ee5ba0bbd1c6d21267325
      size: 20
    - path: process.py
      hash: md5
      md5: 4fef2291690b9644d7f355c7921c95b3
      size: 215
    outs:
    - path: metrics.json
      hash: md5
      md5: 253c1f0c9619dbadc9a1f6843ecd0c7b
      size: 24
    - path: sum.txt
      hash: md5
      md5: 072b030ba126b2f4b2374f342be9ed44
      size: 2



### Example 3 — Push to a remote, then read data back with the Python API

`dvc push` mirrors the cache to a **remote**. Here the remote is just another local
folder, but the workflow is identical for `s3://`, `gs://`, `ssh://`, etc. — only the
URL and credentials differ. Afterward, `dvc.api.read` pulls a tracked file's bytes
straight from the repo without a manual checkout. The cloud-remote variant is gated
behind an env var so the notebook still runs end-to-end offline.

In [4]:
from dvc import api          # Python API; `from ... import` keeps our dvc() helper

# Point DVC at a "remote" (a local dir here; same commands for cloud URLs).
dvc("remote", "add", "-d", "storage", str(REMOTE))
print("push ->", dvc("push"))      # uploads cached data + pipeline outputs

pushed = [p for p in REMOTE.rglob("*") if p.is_file()]
print(f"remote now holds {len(pushed)} content-addressed blob(s) at {REMOTE.name}/")

# Read a DVC-tracked file straight from the repo — no manual `dvc checkout` needed.
content = api.read("data.csv", repo=str(WORK))
print("\ndvc.api.read('data.csv'):")
print(content)

# --- Cloud remote: gated so this notebook runs without credentials ---
# For a real S3 bucket you'd `pip install dvc[s3]` and:
if os.getenv("DVC_S3_BUCKET"):
    dvc("remote", "add", "-d", "s3remote", os.environ["DVC_S3_BUCKET"])
    print(dvc("push"))             # streams to S3 using your AWS creds
else:
    print("\nDVC_S3_BUCKET not set — skipping the cloud push.")
    print("Shape: dvc remote add -d s3remote s3://my-bucket/path && dvc push")

push ->

2 files pushed

remote now holds 2 content-addressed blob(s) at dvc-remote-5q4rxcwu/


dvc.api.read('data.csv'):

id,x
1,10
2,20
3,30



DVC_S3_BUCKET not set — skipping the cloud push.

Shape: dvc remote add -d s3remote s3://my-bucket/path && dvc push

## 6. Gotchas & Pitfalls

- **Committing data instead of the pointer.** After `dvc add`, commit the `*.dvc`
  file (and the updated `.gitignore`) — *not* the data. If the real file is still
  Git-tracked, run `git rm --cached` first; DVC's whole point is keeping bytes out of
  Git.
- **`git pull` without `dvc pull`.** Cloning or pulling restores pointers, not data.
  Your scripts then choke on empty/missing files until you `dvc pull` (or `dvc
  checkout` from a warm cache). Make `dvc pull` part of your setup steps.
- **Editing a DVC-tracked file in place.** DVC doesn't auto-detect edits. After
  changing tracked data you must re-`dvc add` it (or re-`dvc repro` for pipeline
  outputs) to update the hash, then commit the new pointer.
- **A path can't be both a `dvc add` output and a stage output.** Manage each
  artifact one way — either `dvc add` it manually *or* let a pipeline stage produce
  it, never both. DVC errors on the overlap.
- **Cache links and copies.** DVC links files from the cache (reflink/hardlink/
  symlink). On filesystems without reflinks it may fall back to copying (2× space) or
  hardlinks (editing in place silently corrupts the cache). Know your `cache.type`.
- **Forgetting the remote extra.** `dvc push` to S3 fails cryptically if you
  installed plain `dvc` instead of `dvc[s3]`. Install the matching extra.
- **Huge cache, no garbage collection.** Every version's bytes accumulate in the
  cache and remote. Use `dvc gc` (carefully, with `--workspace`/`--all-commits`
  scoping) to reclaim space.
- **Non-deterministic stages.** `dvc repro` decides "changed?" by hashing outputs. A
  stage with random seeds/timestamps looks changed every run and breaks caching —
  pin seeds and avoid embedding timestamps in outputs.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs DVC |
|---|---|---|
| **DVC** | Git-native data/model versioning + reproducible pipelines, any storage backend, no server to run | CLI-centric; large caches need housekeeping; pipeline DSL is one more thing to learn |
| **Git LFS** | Simply storing large binaries in a Git repo | Just storage — no pipelines, metrics, experiments, or cheap branching of data; tied to a Git host's LFS quota |
| **Plain S3/GCS + naming** | Dumping artifacts when versioning is informal | No lineage, no `git checkout`-able reproducibility, no pipeline caching; you reinvent versioning by hand |
| **LakeFS / Pachyderm** | Git-like versioning over *data lakes* / large-scale data pipelines on a cluster | Heavier infra to operate; overkill for repo-scale projects DVC handles with a folder |
| **MLflow / W&B Artifacts** | Experiment tracking with artifact logging inside an MLOps platform | Artifact store, not a Git-integrated data versioner or pipeline engine; often *paired* with DVC, not a replacement |
| **Feature stores (Feast, …)** | Serving curated features online/offline | Different problem (feature serving), not general data/model versioning |

**Rule of thumb:** if your data and models outgrew Git but you still want
`git checkout`-style reproducibility and incremental pipelines without standing up a
server, DVC is the sweet spot. Need only "big files in Git"? Git LFS is simpler.
Operating a data lake at cluster scale? Look at LakeFS/Pachyderm. Tracking
experiments? Pair DVC with MLflow or W&B — they complement rather than compete.

## 8. Resources

- **Official docs** — concepts, commands, full reference: https://dvc.org/doc
- **Get Started / tutorial** — data versioning → pipelines → experiments, hands-on: https://dvc.org/doc/start
- **`dvc.yaml` reference** — stages, deps, outs, foreach, params: https://dvc.org/doc/user-guide/project-structure/dvcyaml-files
- **Python API (`dvc.api`)** — `read`, `open`, `get_url`, `DVCFileSystem`: https://dvc.org/doc/api-reference
- **DVC GitHub** — source, issues, discussions: https://github.com/iterative/dvc